In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision.datasets import MNIST
from torchvision.transforms import transforms

import numpy as np
import matplotlib.pyplot as plt

In [10]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

In [11]:
train_dataset = MNIST(root='./data', train=True, transform=transform)
test_dataset = MNIST(root='./data', train=False, transform=transform)

len(train_dataset)
len(test_dataset)
img, label = train_dataset[0]
img.shape
label

60000

10000

torch.Size([1, 28, 28])

5

In [12]:
train_loader = DataLoader(train_dataset, shuffle=True, batch_size=32)
test_loader = DataLoader(test_dataset, shuffle=False, batch_size=32)

len(train_loader)
len(test_loader)
images, labels = next(iter(train_loader))
images.shape
labels.shape

1875

313

torch.Size([32, 1, 28, 28])

torch.Size([32])

In [13]:
class CNN(nn.Module):
  def __init__(self):
    super(CNN, self).__init__()

    # CNN: capture spatial features
    self.conv1 = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1)
    self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
    self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

    # MLP: classifier head
    self.fc1 = nn.Linear(32 * 7 * 7, 128)
    self.fc2 = nn.Linear(128, 10)
    self.dropout = nn.Dropout(0.25)

  def forward(self, x):
    x = F.relu(self.conv1(x))  # [batch, 1, 28, 28] -> [batch, 16, 28, 28]
    x = self.pool(x)  # 28x28 -> 14x14

    x = F.relu(self.conv2(x))  # [batch, 16, 14, 14] -> [batch, 32, 14, 14]
    x = self.pool(x)  # 14x14 -> 7x7

    # flatten: [batch, 32, 7, 7] -> [batch, 1568]
    x = x.view(x.size(0), -1)

    x = F.relu(self.fc1(x))
    x = self.dropout(x)

    return self.fc2(x)

In [14]:
model = CNN()
total = 0
for name, param in model.named_parameters():
  total += param.numel()
  print(name, tuple(param.shape))
print(f'total params: {total}')

conv1.weight (16, 1, 3, 3)
conv1.bias (16,)
conv2.weight (32, 16, 3, 3)
conv2.bias (32,)
fc1.weight (128, 1568)
fc1.bias (128,)
fc2.weight (10, 128)
fc2.bias (10,)
total params: 206922


In [15]:
model

CNN(
  (conv1): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=1568, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=10, bias=True)
  (dropout): Dropout(p=0.25, inplace=False)
)

In [16]:
images, labels = next(iter(train_loader))
images.shape  # input
model(images).shape  # output

torch.Size([32, 1, 28, 28])

torch.Size([32, 10])